# measure

> Perform profile measurement on cluster images

In [ ]:
# | default_exp euclid.measure

In [ ]:
# | exporti

import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy.cosmology import FlatLambdaCDM

from nicl.autoprof import (
    create_bkgsub_clean_images,
    get_autoprof_info,
    measure_surface_brightness_using_autoprof_isophotes,
    run_autoprof,
)
from nicl.euclid.mask import (
    ICL_BKG_FILTER_SIZE,
    create_combined_nir_mask,
    create_nir_mask,
    create_vis_mask,
)
from nicl.main import configure_logging

In [ ]:
# | hide

from nicl.euclid.data_access import default_data_path

In [ ]:
# | exporti


def mpc_to_pixels(z, pixel_scale_arcsec=0.3, physical_mpc=1.0):
    cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
    arcsec_per_kpc = cosmo.arcsec_per_kpc_proper(z).value  # arcsec/kpc
    arcsec_per_mpc = arcsec_per_kpc * 1000  # arcsec/Mpc
    pixels_per_mpc = arcsec_per_mpc / pixel_scale_arcsec
    return pixels_per_mpc * physical_mpc

In [ ]:
# | export


class ClusterPipeline:
    def __init__(
        self,
        image_dir: Path | str,  # directory containing the measurement images
        output_dir: Path | str,  # directory to save the output files
        cluster_id: str,  # cluster ID, for labelling the output files
        cluster_z: float
        | None,  # cluster redshift, for calculating the background box size; can be None if box_size is provided
        filters: str
        | list[
            str
        ],  # filters in which to measure isophotes and/or surface brightness profiles
        image_label: str = "",  # to identify if a non-standard images have been used (useful during tests)
        mask_filter: str
        | None = None,  # filter of the mask that will be applied to the measurement image; if None, no mask is applied
        pixelscale: float = 0.3,  # pixel scale of the images in arcsec/pixel
        isophotes_geometric_factor: float = 0.4,  # geometric factor for increasing isophote radii
        isophotes_regularization_scale: float = 1,  # scale factor for the regularisation of the isophotes
        bcg_pos: SkyCoord
        | None = None,  # BCG position; if None, the BCG position is taken from the cluster_info_table
        box_size: int
        | bool
        | None = None,  # the background box size in pixels; if None, 1 Mpc is used; if False, no background is estimated
        isophotes_filter: str
        | None = None,  # use the isophotes measured in this filter
        isophotes_label: str = "",  # to identify if non-default isophote parameters have been used (useful during tests)
        external_mask_filename: Path
        | str
        | None = None,  # use this mask instead of creating a new one from the measurement image
        external_bkg_mask_filename: Path
        | str
        | None = None,  # use this background mask instead of creating a new one from the measurement image
        mask_label: str = "",  # to identify if non-default mask parameters have been used (useful during tests)
        noise_file_dir: Path
        | str
        | None = None,  # use the noise estimates in this directory
        noise_field: str
        | None = None,  # use the noise properties for this field, e.g. EDFS, EDFF or EDFN
    ):
        self.valid_filters = ["H", "J", "Y", "YJH", "VIS"]
        self.valid_mask_filters = self.valid_filters + [None]
        self.logger = logging.getLogger(__name__)
        self.image_dir = self._validate_path(image_dir, is_dir=True)
        self.outdir = Path(output_dir)
        self.cluster_id = cluster_id
        self.filters = self._validate_filters(filters)
        self.image_label = image_label
        self.mask_filter = self._validate_mask_filter(mask_filter)
        self.cluster_z = cluster_z
        self.pixelscale = pixelscale
        self.isophotes_geometric_factor = isophotes_geometric_factor
        self.isophotes_regularization_scale = isophotes_regularization_scale
        self.bcg_pos = self._validate_bcg_pos(bcg_pos)
        self.isophotes_filter = self._validate_isophotes_filter(isophotes_filter)
        self.isophotes_label = isophotes_label
        self.external_mask_filename = self._validate_path(external_mask_filename)
        self.external_bkg_mask_filename = self._validate_path(
            external_bkg_mask_filename
        )
        if mask_label:
            self.mask_label = mask_label
        else:
            self.mask_label = self.image_label
        self.noise_file_dir = self._validate_path(noise_file_dir, is_dir=True)
        self.noise_field = noise_field
        self._set_box_size(box_size)
        self.cluster_output_dir = self.outdir / self.cluster_id

    def _validate_filters(self, filters):
        filters = [filters] if isinstance(filters, str) else filters
        if not set(filters).issubset(self.valid_filters):
            raise ValueError(
                f"Unknown filters: {filters}. Valid filters are {self.valid_filters}."
            )
        return filters

    def _validate_mask_filter(self, mask_filter):
        if mask_filter not in self.valid_mask_filters:
            raise ValueError(
                f"Unknown mask filter: {mask_filter}. Valid mask filters are {self.valid_mask_filters}."
            )
        if mask_filter is not None:
            for filter in self.filters:
                filters_match = filter == mask_filter
                filters_match |= filter in ("Y", "J", "H") and mask_filter == "YJH"
                if not filters_match:
                    raise ValueError(
                        f"Mask filter {mask_filter} does not match filter {filter}."
                    )
        return mask_filter

    def _validate_isophotes_filter(self, isophotes_filter):
        if isophotes_filter not in [None] + self.valid_filters:
            raise ValueError(
                f"Unknown isophotes filter: {isophotes_filter}. Valid isophotes filters are {self.valid_filters}."
            )
        return isophotes_filter

    def _validate_bcg_pos(self, bcg_pos):
        if isinstance(bcg_pos, str):
            bcg_pos = SkyCoord(bcg_pos)
            self.logger.debug(f"BCG position parsed from passed argument: {bcg_pos}")
        elif isinstance(bcg_pos, SkyCoord):
            self.logger.debug(f"BCG position is taken from passed argument: {bcg_pos}")
        elif bcg_pos is None:
            self.logger.debug("BCG position will be taken as centre of the image.")
        else:
            raise ValueError(f"Invalid BCG position: {bcg_pos}")
        return bcg_pos

    def _validate_path(self, path, is_dir=False):
        if path is not None:
            path = Path(path)
            if not path.exists():
                raise FileNotFoundError(f"File {path} not found.")
            if is_dir and not path.is_dir():
                raise ValueError(f"Path {path} is not a directory.")
            if not is_dir and not path.is_file():
                raise ValueError(f"Path {path} is not a file.")
        return path

    def _set_box_size(self, box_size=None):
        if box_size is None or box_size is True:
            box_size = int(
                mpc_to_pixels(z=self.cluster_z, pixel_scale_arcsec=self.pixelscale)
            )
            self.logger.debug(
                f"Using a background box size of {box_size} pixels, corresponding to 1 Mpc at z={self.cluster_z} with {self.pixelscale} arcsec/pixel."
            )
        elif box_size is False:
            self.logger.debug("No background subtraction will be performed.")
        else:
            self.logger.debug(
                f"Using background box size from passed argument: {box_size} pixels"
            )
        self.box_size = box_size

    def _get_mask_name(self, include_filter=False):
        mask_name = f"{self.cluster_id}"
        if self.mask_label:
            mask_name += f"_{self.mask_label}"
        if include_filter:
            mask_name += f"_{self.mask_filter}"
        return mask_name

    def _get_isophotes_name(self, filter, mask_filter=None):
        name = f"{self.cluster_id}"
        if self.mask_filter is None:
            name += "-nomask"
        else:
            name += "-mask"
            if self.mask_label:
                name += f"_{self.mask_label}"
            if mask_filter is not None:
                name += f"_{mask_filter}"
            else:
                name += f"_{self.mask_filter}"
        name += "-iso"
        if self.isophotes_label:
            name += f"_{self.isophotes_label}"
        if self.isophotes_filter is None:
            name += f"_{filter}"
        else:
            name += f"_{self.isophotes_filter}"
        return name

    def _get_photometry_name(self, isophotes_name, filter):
        name = isophotes_name
        name += "-phot"
        if self.image_label:
            name += f"_{self.image_label}"
        name += f"_{filter}"
        return name

    def _get_masks(self):
        if any(
            [
                self.mask_filter,
                self.external_mask_filename,
                self.external_bkg_mask_filename,
            ]
        ):
            self.logger.info(f"Getting masks for {self.cluster_id}")
            mask_path = self._get_measurement_mask()
            bkg_mask_path = self._get_background_mask()
            self.logger.info(f"Measurement mask is {mask_path}")
            self.logger.info(f"Background mask is {bkg_mask_path}")
        else:
            mask_path = bkg_mask_path = None
        return mask_path, bkg_mask_path

    def _get_measurement_mask(self):
        mask_name = self._get_mask_name(include_filter=True)
        if self.external_mask_filename is not None:
            mask_path = Path(self.external_mask_filename)
        elif self.mask_filter is None:
            mask_path = None
        else:
            mask_path = self.cluster_output_dir / f"{mask_name}_measurement_mask.fits"
        if mask_path is not None and not mask_path.exists():
            raise FileNotFoundError(
                f"Expected measurement mask not found at {mask_path}"
            )
        return mask_path

    def _get_background_mask(self):
        mask_name = self._get_mask_name(include_filter=True)
        if self.external_bkg_mask_filename is not None:
            bkg_mask_path = Path(self.external_bkg_mask_filename)
        elif self.mask_filter is None:
            bkg_mask_path = None
        else:
            bkg_mask_path = (
                self.cluster_output_dir / f"{mask_name}_background_mask.fits"
            )
        if bkg_mask_path is not None and not bkg_mask_path.exists():
            raise FileNotFoundError(
                f"Expected background mask not found at {bkg_mask_path}"
            )
        return bkg_mask_path

    def create_masks(self):
        try:
            self._get_masks()
        except FileNotFoundError:
            self._create_masks()

    def _create_masks(self):
        self._create_output_dir()
        if self.mask_filter == "YJH":
            self._create_combined_nir_image_and_mask()
        elif self.mask_filter == "VIS":
            self._create_vis_mask()
        else:
            self._create_nir_mask(self.mask_filter)

    def _create_combined_nir_image_and_mask(self):
        mask_name = self._get_mask_name()
        self.logger.info(f"Creating combined NIR image and masks for {mask_name}...")
        image_filenames = self._get_nir_image_filenames()
        create_combined_nir_mask(
            image_filenames["H"],
            image_filenames["J"],
            image_filenames["Y"],
            centre_pos=self.bcg_pos,
            label=mask_name,
            output_dir=self.cluster_output_dir,
            icl_bkg_box_size=self.box_size,
            nir_stack_bkg_box_size=self.box_size,
        )
        self.logger.info("Created combined NIR image and masks.")

    def _create_vis_mask(self):
        mask_name = self._get_mask_name()
        self.logger.info(f"Creating new VIS masks for {mask_name}...")
        image_filename = self._get_vis_image_filenames()
        create_vis_mask(
            image_filename,
            centre_pos=self.bcg_pos,
            label=mask_name,
            output_dir=self.cluster_output_dir,
            icl_bkg_box_size=self.box_size,
        )
        self.logger.info("Created VIS masks.")

    def _create_nir_mask(self, filter):
        mask_name = self._get_mask_name()
        self.logger.info(f"Creating new NIR masks for {mask_name}...")
        image_filename = self._get_nir_image_filenames(filter)
        create_nir_mask(
            image_filename,
            filter=filter,
            centre_pos=self.bcg_pos,
            label=mask_name,
            output_dir=self.cluster_output_dir,
            icl_bkg_box_size=self.box_size,
        )
        self.logger.info("Created NIR masks.")

    def _get_nir_image_filenames(self, filter=None):
        if filter is None:
            image_filenames = {}
            for filter in ["H", "J", "Y"]:
                image_filenames[filter] = self._get_image_filename(filter)
        else:
            image_filenames = self._get_image_filename(filter)
        return image_filenames

    def _get_vis_image_filenames(self):
        return self._get_image_filename("VIS")

    def _get_image_filename(self, filter):
        suffix = f"[_-]{self.image_label}" if self.image_label else ""
        if filter == "YJH":
            search_dir = self.cluster_output_dir
            pattern = f"{self.cluster_id}{suffix}_{filter}.fits"
            self.logger.debug(f"Searching for {pattern} in {search_dir}")
            matches = list(search_dir.glob(pattern))
        else:
            search_dir = self.image_dir
            pattern = f"{self.cluster_id}[_-]{filter}{suffix}.fits"
            self.logger.debug(f"Searching for {pattern} in {search_dir}")
            matches = list(search_dir.glob(pattern))
            pattern = f"*[_-]{filter}[_-]{self.cluster_id}{suffix}.fits"
            self.logger.debug(f"Searching for {pattern} in {search_dir}")
            matches += list(search_dir.glob(pattern))
            pattern = f"*[_-]{filter}[_-]*[_-]{self.cluster_id}{suffix}.fits"
            self.logger.debug(f"Searching for {pattern} in {search_dir}")
            matches += list(search_dir.glob(pattern))
        if len(matches) == 0:
            raise FileNotFoundError(
                f"No image found for {self.cluster_id} and filter {filter} in {search_dir}"
            )
        if len(matches) > 1:
            raise FileNotFoundError(
                f"Multiple images found for {self.cluster_id} and filter {filter} in {search_dir}"
            )
        return str(matches[0])

    def _create_output_dir(self):
        self.cluster_output_dir.mkdir(parents=True, exist_ok=True)

    def measure_isophotes(self, filter, mask_filter=None):
        self.logger.info(
            f"Getting AutoProf isophotes for {self.cluster_id} in {filter} band"
        )

        output_name = self._get_isophotes_name(filter, mask_filter)
        autoprof_results_dir = self.cluster_output_dir / "autoprof_results"

        output_filename = autoprof_results_dir / f"{output_name}.prof"

        if self.isophotes_filter is not None and self.isophotes_filter != filter:
            self.logger.info(f"Attempting to use existing {output_name} isophotes")
            if not output_filename.exists():
                raise FileNotFoundError(
                    f"Expected isophotes file {output_filename} not found."
                )
            self.logger.info(f"Isophotes file {output_filename} found.")
            return output_filename
        elif output_filename.exists():
            self.logger.info(f"Autoprof output file {output_filename} already exists.")
            self.logger.info("Skipping Autoprof run.")
            return output_filename

        self.logger.debug(f"Output name is {output_name}")
        image_filename = self._get_image_filename(filter)
        self.logger.debug(f"Image filename is {image_filename}")
        self.logger.debug(f"Cluster output directory is {self.cluster_output_dir}")
        autoprof_results_dir.mkdir(parents=True, exist_ok=True)

        mask_path, bkg_mask_path = self._get_masks()

        temp_dir = self.cluster_output_dir / "tmp/autoprof"

        self.logger.info(
            f"Subtracting background with box size {self.box_size} pixels and cleaning NaNs"
        )

        cleaned_image_filename = create_bkgsub_clean_images(
            image_filenames=image_filename,
            mask_filename=bkg_mask_path,
            output_dir=temp_dir,
            output_background_dir=self.cluster_output_dir,
            box_size=self.box_size,
            filter_size=ICL_BKG_FILTER_SIZE,
            clean_nans=True,
        )

        self.logger.info(f"Autoprof running on {cleaned_image_filename}")
        # If the BCG position is not provided, we assume it is the centre of the image
        fixed_centre = self.bcg_pos if self.bcg_pos is not None else True
        run_autoprof(
            name=output_name,
            image_filename=cleaned_image_filename,
            mask_filename=mask_path,
            out_dir=autoprof_results_dir,
            gscale=self.isophotes_geometric_factor,
            fixed_centre=fixed_centre,
            regularize_scale=self.isophotes_regularization_scale,
        )
        self._clean_autoprof_results(autoprof_results_dir, cleaned_image_filename)
        return output_filename

    def _clean_autoprof_results(
        self, autoprof_results_dir, cleaned_image_filename=None
    ):
        always_keep = []
        jpg_keywords_to_keep = [
            "mask_",
            "initialize_ellipse_",
            "photometry_",
            "photometry_ellipse_",
        ]
        valid_suffixes_to_keep = [".prof", ".aux", ".py", ".log"]

        for f in autoprof_results_dir.iterdir():
            if f.is_file():
                name = f.name
                if name in always_keep:
                    continue
                if any(name.endswith(suffix) for suffix in valid_suffixes_to_keep):
                    continue
                if name.endswith(".jpg") and any(
                    kw in name for kw in jpg_keywords_to_keep
                ):
                    continue
                try:
                    f.unlink()
                    self.logger.debug(f"Deleted: {name}")
                except Exception as e:
                    self.logger.warning(f"Could not delete {name}: {e}")

        if cleaned_image_filename is not None:
            try:
                cleaned_image_filename.unlink()
                self.logger.debug(f"Deleted: {name}")
            except Exception as e:
                self.logger.warning(f"Could not delete {cleaned_image_filename}: {e}")

    def measure_photometry(
        self, filter, isophotes_filter=None, isophotes_mask_filter=None
    ):
        self.logger.info(f"Subtracting background with box size {self.box_size} pixels")
        image_filename = self._get_image_filename(filter)
        mask_path, bkg_mask_path = self._get_masks()
        temp_dir = self.cluster_output_dir / "tmp/sb_profile"
        if isophotes_filter is None:
            isophotes_filter = filter
        if isophotes_mask_filter is None:
            isophotes_mask_filter = "VIS" if isophotes_filter == "VIS" else "YJH"
        isophotes_name = self._get_isophotes_name(
            isophotes_filter, mask_filter=isophotes_mask_filter
        )
        label = self._get_photometry_name(isophotes_name, filter)
        autoprof_filename = (
            self.cluster_output_dir / "autoprof_results" / f"{isophotes_name}.prof"
        )

        bkgsub_image_filename = create_bkgsub_clean_images(
            image_filenames=image_filename,
            mask_filename=bkg_mask_path,
            output_dir=temp_dir,
            box_size=self.box_size,
            filter_size=ICL_BKG_FILTER_SIZE,
            clean_nans=False,
        )

        self.logger.info(
            f"Extracting surface brightness profile for {self.cluster_id} in {filter}."
        )
        self.logger.info(
            f"Using {self.mask_filter} band mask and isophotes from {autoprof_filename.name}."
        )
        self.logger.info(f"Label for the output file is {label}.")

        flux_measurements, profile, problematic_annuli = (
            measure_surface_brightness_using_autoprof_isophotes(
                image_filename=bkgsub_image_filename,
                object_mask_filename=mask_path
                if self.mask_filter is not None
                else None,
                profile_filename=autoprof_filename,
                centre=self.bcg_pos,
            )
        )

        if not self.keep_temp_files:
            bkgsub_image_filename.unlink()

        flux_outfile = self.cluster_output_dir / f"{label}-profile_measurements.csv"
        flux_measurements.to_csv(flux_outfile, index=False)
        profile_df = self._merge_profiles(filter, autoprof_filename, flux_outfile)
        return profile_df, flux_outfile

    def _get_noise_profile(self, filter):
        # Loading noise file and finding appropriate box sized one...
        if self.noise_file_dir is not None:
            if self.field:
                print(f"Noise field is {self.field}")
                noise_bkg_sizes = np.array([450, 500, 550, 650, 750, 1000, 1800, 2350])
                idx = (np.abs(noise_bkg_sizes - self.box_size)).argmin()
                closest_bs = noise_bkg_sizes[idx]
                self.logger.info(
                    f"1 Mpc at cluster corresponds to {self.box_size} and closest box size used in noise measurements is: {closest_bs}"
                )

                noise_file = (
                    self.noise_file_dir
                    / f"{self.field}_Skypatch_bs{closest_bs}_{filter}_noise_measurements.csv"
                )

                if noise_file.exists():
                    noise_df = pd.read_csv(noise_file)
                else:
                    raise FileNotFoundError(f"Noise file {noise_file.name} not found.")
            else:
                raise ValueError(
                    "Field needs to be provided to ensure correct noise properties are added to the cluster profile file."
                )
        else:
            noise_df = None
        return noise_df

    def _merge_profiles(
        self, filter, autoprof_filename, sb_profile_filename, include_noise=True
    ):
        self.logger.info(
            f"Merging isophotes, profile and noise for {self.cluster_id} in {filter}."
        )
        # Reading .prof file to merge isophote, annulus method results and the noise curves
        with open(autoprof_filename) as f:
            units_line = f.readline().strip()
            column_line = f.readline().strip()
            column_list = column_line.split(",")
            profile_df = pd.read_csv(f, names=column_list)
            unit_list = units_line.lstrip("#").split(",")
            profile_df_units = {c: u for c, u in zip(column_list, unit_list)}

        if self.isophotes_filter != filter:
            # Removing AutoProf photometry as it is for a different filter
            profile_df = profile_df.drop(columns=["I", "I_e", "totflux", "totflux_e"])
        else:
            # Adding background level into the AutoProf profile
            autoprof_background_level = get_autoprof_info(autoprof_filename)[
                "background_level"
            ]
            self.logger.info(
                f"AutoProf background level is {autoprof_background_level:.5f}"
            )
            if autoprof_background_level:
                self.logger.info("Adding background level to the AutoProf profile.")
                profile_df["I"] = profile_df["I"] + autoprof_background_level

        flux_measurements = pd.read_csv(sb_profile_filename)
        profile_df["SMA_annulus_centre"] = flux_measurements[
            "SMA_annulus_centre_arcsec"
        ]
        profile_df_units["SMA_annulus_centre"] = "arcsec"
        profile_df["Median_flux_annulus"] = flux_measurements[
            "Clipped_median_flux_annulus"
        ]
        profile_df_units["Median_flux_annulus"] = "flux*arcsec^-2"

        if include_noise:
            noise_df = self._get_noise_profile(filter)
            if noise_df is not None:
                # Adding noise columns
                profile_df["SMA_annulus_centre_noise"] = noise_df[
                    "SMA_annulus_centre_arcsec"
                ]
                profile_df_units["SMA_annulus_centre_noise"] = "arcsec"
                profile_df["MAD_median_clipped_flux_noise"] = noise_df[
                    "MAD_Median_Clipped_Flux"
                ]
                profile_df_units["MAD_median_clipped_flux_noise"] = "flux*arcsec^-2"
                profile_df["MAD_bkg_subtracted_flux_noise"] = noise_df[
                    "MAD_Bkg_Subtracted_Flux"
                ]
                profile_df_units["MAD_bkg_subtracted_flux_noise"] = "flux*arcsec^-2"
            else:
                include_noise = False
        # Cleaning nans for Autoprof, it hates nans
        profile_df.fillna(0, inplace=True)

        label = self._get_photometry_name(autoprof_filename.stem, filter)

        merged_prof_path = self.cluster_output_dir / f"{label}_merged.prof"
        with open(merged_prof_path, "w") as f:
            column_list = profile_df.columns
            unit_list = [profile_df_units[c] for c in column_list]
            f.write("#" + ",".join(unit_list) + "\n")
            f.write(",".join(column_list) + "\n")
            profile_df.to_csv(f, index=False, header=False)

        self.logger.info(
            f"Final updated .prof file with surface brightness profile{' and noise ' if include_noise else ' '}written successfully."
        )
        return profile_df

    def _plot_profiles(self, profile_df, output_name):
        fig, ax = plt.subplots(1, 1)
        ax.plot(
            profile_df["SMA_annulus_centre"],
            profile_df["Median_flux_annulus"],
            color="tab:green",
            label="Median Flux Annuli",
        )
        if "I" in profile_df.columns:
            ax.plot(
                profile_df["R"],
                profile_df["I"],
                color="tab:blue",
                label="Median Flux Isophote",
            )
        ax.set_ylabel(r"Surface Brightness ($\rm flux \ arcsec^{-2}$)")
        ax.set_xlabel("semi-major axis (arcsec)")
        ax.set_yscale("log")
        ax.set_xscale("log")
        fig_path = self.cluster_output_dir / f"SB_comparisons_{output_name}.pdf"
        plt.legend(loc="upper right", fontsize=10)
        plt.savefig(fig_path)
        plt.close(fig)

        fig, ax = plt.subplots(1, 2, figsize=(9, 4))
        ax[0].errorbar(
            profile_df["R"],
            profile_df["ellip"],
            yerr=profile_df["ellip_e"],
            color="tab:blue",
        )
        ax[0].set_ylim(0, 1)
        ax[0].set_ylabel("Ellipticity")
        ax[1].errorbar(
            profile_df["R"],
            profile_df["pa"],
            yerr=profile_df["pa_e"],
            color="tab:blue",
        )
        ax[1].set_ylim(0, 180)
        ax[1].set_ylabel("Position Angle [deg]")

        for a in fig.get_axes():
            a.set_xlabel("semi-major axis (arcsec)")
            a.set_xscale("log")

        fig_path = self.cluster_output_dir / f"Shapes_{output_name}.pdf"
        plt.savefig(fig_path)
        plt.close(fig)

    def run(self, filter=None):
        if filter is None:
            for filter in self.filters:
                self.run_filter(filter)
        else:
            self.run_filter(filter)

    def run_filter(self, filter, extract_sb_profile=True, plot_profiles=True):
        self.logger.info(
            f"Cluster id: {self.cluster_id}, redshift: {self.cluster_z}, BCG coords: {self.bcg_pos}, box size: {self.box_size}"
        )
        self.logger.info(
            f"Processing {filter} band image with {self.mask_filter} band mask."
        )
        # If we are running on the YJH image, we need to ensure it is created. Currently, this is done when creating
        # the YJH mask, so we need to force this even if no masking is being done.
        force_YJH = filter == "YJH" and self.mask_filter != "YJH"
        if force_YJH:
            saved_mask_filter = self.mask_filter
            self.mask_filter = "YJH"
        self.create_masks()
        if force_YJH:
            self.mask_filter = saved_mask_filter
        self.measure_isophotes(filter)
        if extract_sb_profile:
            profile_df, sb_profile_filename = self.measure_photometry(filter)
            # FIXME: The plotting code should find and read the correct isophotes file independently
            if plot_profiles:
                self._plot_profiles(profile_df, sb_profile_filename.stem)

## Examples

In [ ]:
configure_logging(logfile=default_data_path("test_measure") / "measure.log")
configure_logging(name="__main__", level="DEBUG")
configure_logging(name="nicl.euclid.mask", level="DEBUG")
configure_logging(name="nicl.mask", level="DEBUG")

<Logger nicl.mask (DEBUG)>

### Test clusters

In [ ]:
for name in ["basic_test", "varying_background", "real_background"]:
    image_path = default_data_path("test_images") / name
    out_path = default_data_path("test_measure") / name
    filters = [
        "YJH",
        ["Y", "J", "H"],
        ["Y", "J", "H"],
        "VIS",
    ]
    mask_filters = [
        "YJH",
        "YJH",
        "YJH",
        "VIS",
    ]
    isophote_filters = [
        None,
        None,
        "YJH",
        None,
    ]
    for filter, mask_filter, isophote_filter in zip(
        filters, mask_filters, isophote_filters
    ):
        pipeline = ClusterPipeline(
            image_dir=image_path,
            output_dir=out_path,
            cluster_id="cluster",
            cluster_z=0.1,
            filters=filter,
            mask_filter=mask_filter,
            isophotes_filter=isophote_filter,
        )
        pipeline.run()
        if name == "basic_test":
            pipeline = ClusterPipeline(
                image_dir=image_path,
                image_label="no_noise",
                output_dir=out_path,
                cluster_id="cluster",
                cluster_z=0.1,
                filters=filter,
                mask_filter=None,
                isophotes_filter=isophote_filter,
            )
            pipeline.run()

### Mock clusters

In [ ]:
# cluster_ids = ["cluster0"]
# outdir_base = default_data_path("Q1_R1_clusters_v0.7/analysis/TK/mock_tests")
# outdir_base = Path("/home/ppztk1/Erosita/Outputs_Clusters/")
# image_base = Path("/home/ppzjbg/JGM_Tests/Mock_Images/")

# process_cluster_pipeline(
#     image_dir=image_base,
#     outdir=outdir_base,
#     cluster_ids=cluster_ids,
#     cluster_z=0.1,
#     filters="H",
#     mock_image=True,
#     masking=True,
#     mask_filter="YJH",
#     run_autoprof_function=True,
#     SB_extraction=True,
#     show_profiles=True,
#     image_label="skypatch_field",
# )

In [ ]:
# cluster_ids = ["cluster0", "cluster1", "cluster2"]
# outdir_base = default_data_path("Q1_R1_clusters_v0.7/analysis/TK/mocks")
# image_base = Path("/home/ppzjbg/JGM_Tests/Mock_Images/")

# filters = ["VIS"]

# for filter in filters:
#     if filter in ["H", "J", "Y", "VIS"]:
#         image_dir = image_base
#         outdir = outdir_base

#         process_cluster_pipeline(
#             image_dir=image_dir,
#             outdir=outdir,
#             cluster_ids=cluster_ids,
#             cluster_z=0.1,
#             filters=filter,
#             mock_image=True,
#             masking=True,
#             mask_filter="YJH",
#             run_autoprof_function=True,
#             SB_extraction=True,
#             show_profiles=True,
#             image_label="skypatch_field",
#         )

#     elif filter == "YJH":
#         for clid in cluster_ids:
#             image_dir = Path(f"/home/ppztk1/Erosita/Outputs_Clusters/{clid}/")
#             outdir = outdir_base

#             process_cluster_pipeline(
#                 image_dir=image_dir,
#                 outdir=outdir,
#                 cluster_ids=[clid],
#                 filters=filter,
#                 cluster_z=0.1,
#                 mock_image=True,
#                 masking=True,
#                 mask_filter="YJH",
#                 run_autoprof_function=True,
#                 SB_extraction=True,
#                 show_profiles=True,
#                 image_label="skypatch_field",
#             )

## Real clusters

In [ ]:
# cluster_ids = ["EDFS_eRASS_2"]

# image_dir = Path("/home/ppztk1/euclid_data/Q1_R1_clusters_v0.7/TK/")
# noise_dir = Path("/home/ppztk1/Erosita/Outputs_Clusters/background_skypatch/")
# table = pd.read_csv("/home/ppztk1/Erosita/Tables/Erosita_EDFS_LargeSample.csv")

# output_dir = Path("/home/ppztk1/Erosita/Outputs_Clusters/")


# filters = ["H", "J", "Y"]
# process_cluster_pipeline(
#     # image_dir=image_dir,
#     outdir=output_dir,
#     cluster_ids=cluster_ids,
#     cluster_info_table=table,
#     filters=filters,
#     mock_image=False,
#     masking=True,
#     mask_filter="YJH",
#     run_autoprof_function=False,
#     SB_extraction=True,
#     show_profiles=True,
#     noise_file_path=noise_dir,
# )


# filters = ["YJH"]
# process_cluster_pipeline(
#     image_dir=image_dir,
#     outdir=output_dir,
#     cluster_ids=cluster_ids,
#     cluster_info_table=table,
#     filters=filters,
#     mock_image=False,
#     masking=True,
#     mask_filter="YJH",
#     run_autoprof_function=True,
#     SB_extraction=True,
#     show_profiles=True,
#     noise_file_path=noise_dir,
# )

# filters = ["VIS"]
# process_cluster_pipeline(
#     image_dir=image_dir,
#     outdir=output_dir,
#     cluster_ids=cluster_ids,
#     cluster_info_table=table,
#     filters=filters,
#     mock_image=False,
#     masking=True,
#     mask_filter="VIS",
#     run_autoprof_function=True,
#     SB_extraction=True,
#     show_profiles=True,
#     noise_file_path=noise_dir,
# )

# filters = ["H", "J", "Y"]
# process_cluster_pipeline(
#     image_dir=image_dir,
#     outdir=output_dir,
#     cluster_ids=cluster_ids,
#     cluster_info_table=table,
#     filters=filters,
#     mock_image=False,
#     masking=True,
#     mask_filter="YJH",
#     run_autoprof_function=True,
#     SB_extraction=True,
#     noise_file_path=noise_dir,
#     show_profiles=True,
#     forced_photometry=True,
#     forced_profile_filter="YJH",
#     forced_profile_path="/home/ppztk1/Erosita/Outputs_Clusters/EDFS_eRASS_56/autoprof_results/EDFS_eRASS_56_YJH_YJH_YJH.prof",
# )


# filters = ["VIS"]
# process_cluster_pipeline(
#     image_dir=image_dir,
#     outdir=output_dir,
#     cluster_ids=cluster_ids,
#     cluster_info_table=table,
#     filters=filters,
#     mock_image=False,
#     masking=True,
#     mask_filter="VIS",
#     run_autoprof_function=True,
#     SB_extraction=True,
#     noise_file_path=noise_dir,
#     show_profiles=True,
#     forced_photometry=True,
#     forced_profile_filter="YJH",
#     forced_profile_path="/home/ppztk1/Erosita/Outputs_Clusters/EDFS_eRASS_56/autoprof_results/EDFS_eRASS_56_YJH_YJH_YJH.prof",
# )